In [490]:
import pandas as pd
import matplotlib.pyplot as plt
import logomaker
import os
import scipy.stats
import matplotlib.gridspec as gridspec
import glob
import sys
import numpy as np
import collections

sys.path.append('/data/gralak/meSMiLEseq_github/meSMiLEseq')
import utils

In [491]:
#Defining dependencies and stuff needed for the script to running properly
def _dictionary():
    return {
        'exp1': 'SmSAG01',
        'exp01': 'SmSAG01',
        'exp2': 'SmSAG02',
        'exp02': 'SmSAG02',
        'exp3': 'SmSAG03',
        'exp03': 'SmSAG03',
        'exp4': 'SmSAG04',
        'exp04': 'SmSAG04',
        'exp5': 'SmSAG05',
        'exp05': 'SmSAG05',
        'exp6': 'SmSAG06',
        'exp06': 'SmSAG06',
        'exp7': 'SmSAG07',
        'exp07': 'SmSAG07',
        'exp8': 'SmSAG08',
        'exp08': 'SmSAG08',
        'exp9': 'SmSAG09',
        'exp09': 'SmSAG09',
        'exp10': 'SmSAG10',
        'exp11': 'SmSAG11',
        'exp12': 'SmSAG12',
        'exp13': 'SmSAG13',
        'exp14': 'SmSAG14',
        'exp15': 'SmSAG15',
        'exp16': 'SmSAG16',
        'exp17': 'SmSAG17',
        'exp18': 'SmSAG18',
        'exp19': 'SmSAG19',
        'exp20': 'SmSAG20',
        'exp21': 'SmSAG21',
        'exp22': 'SmSAG22',
        'exp23': 'SmSAG23',
        'exp24': 'SmSAG24',
        '1': 'BC1',
        '2': 'BC2',
        '3': 'BC3',
        '4': 'BC4',
        '5': 'BC5',
        '6': 'BC6',
        '7': 'BC7',
        '8': 'BC8',
        '9': 'BC9',
        '10': 'BC10',
        '11': 'BC11',
        '12': 'BC12',
        'input1_fwd/rev': '20220315_input',
        'input2_fwd/rev': '20220915_input',
        'input3_fwd/rev': '20230228_input',
        'input4_fwd/rev': '20230503_input'
}
metadata_dict = _dictionary()

def assign_mBC(input_spec):
    """function accepts a character string as input_spec, so input1_fwd/rev etc. 
    Returns as first element methylated Barcode, as second the unmethylated counterpart, and the path to the data."""
    if input_spec == 'input1_fwd/rev':
        methylated_BC = "AGTA"
        unmethylated_BC = "GAGT"
        input_path = '/home/gralak/updepla/users/gralak/NAS2/SmileSeq_paper/SmileSeq_experiments/inputs/20220315_input/00_read_in_data/output/'
    elif input_spec == 'input2_fwd/rev':
        methylated_BC = "AGTA"
        unmethylated_BC = "GAAT"
        input_path = '/home/gralak/updepla/users/gralak/NAS2/SmileSeq_paper/SmileSeq_experiments/inputs/20220915_input/00_read_in_data/output/'
    elif input_spec == 'input3_fwd/rev':
        methylated_BC = "AGTA"
        unmethylated_BC = "GAAT"
        input_path = '/home/gralak/updepla/users/gralak/NAS2/SmileSeq_paper/SmileSeq_experiments/inputs/20230228_input/00_read_in_data/output/'
    elif input_spec == 'input4_fwd/rev':
        methylated_BC = "AGTA"
        unmethylated_BC = "GAAT"
        input_path = '/home/gralak/updepla/users/gralak/NAS2/SmileSeq_paper/SmileSeq_experiments/inputs/20230503_input/00_read_in_data/output/'
    
    return (methylated_BC, unmethylated_BC, input_path)




In [492]:
def write_sectioned_csv(path, sections):
    """
    sections: list of (title, obj) where obj is a pandas DataFrame,
              a 2D list/array, or anything pd.DataFrame() can handle.
    """
    with open(path, "w", newline="") as f:
        for i, (title, obj) in enumerate(sections):
            f.write(f"{title}\n")                 # section header (single cell)
            df = obj if isinstance(obj, pd.DataFrame) else pd.DataFrame(obj)
            df.to_csv(f)                          # include index & header by default
            if i < len(sections) - 1:
                f.write("\n")                     # blank line as separator

In [493]:
metadata = pd.read_csv('/data/gralak/meSMiLEseq_github/meSMiLEseq/metadata.csv')
latest_curation = pd.read_csv('/data/gralak/meSMiLEseq_github/meSMiLEseq/TF_with_motifs_latest_curation.csv')
TFs = os.listdir('/home/gralak/updepla/users/gralak/SmileSeq_paper/meSMiLEseq_motifs_and_scatterplots_for_publication')
filtered_TFs = [tf for tf in TFs if not tf.endswith('.csv')]

In [494]:
filtered_TFs = ['PRDM10_DBD', 'ZNF614_FL', 'ZNF680_FL']

In [495]:
for protein in filtered_TFs:
    TF = protein
    data_path = f'/home/gralak/updepla/users/gralak/SmileSeq_paper/meSMiLEseq_motifs_and_scatterplots_for_publication/{TF}/'

    joint_matrices_path = os.path.join(data_path, 'joint/matrices/')
    sep_matrices_path = os.path.join(data_path, 'separated/matrices/')    

    try:
        meth_consensus = pd.read_csv(os.path.join(data_path, 'methylated_consensus_ppm_logo.csv'))
    except FileNotFoundError:
        meth_consensus = None

    try:
        unmeth_consensus = pd.read_csv(os.path.join(data_path, 'unmethylated_consensus_ppm_logo.csv'))
    except FileNotFoundError:
        unmeth_consensus = None

    div_series = pd.read_csv(os.path.join(data_path, 'divergence_series.csv'))



    files_sep = os.listdir(sep_matrices_path)
    meths = [f for f in files_sep if f.endswith('_methylated_bindingmode_2.csv')]
    unmeths = [f for f in files_sep if f.endswith('_unmethylated_bindingmode_2.csv')]
    #meths = [f for f in files_sep if f.endswith('_methylated_bindingmode_1.csv')]
    #unmeths = [f for f in files_sep if f.endswith('_unmethylated_bindingmode_1.csv')]

    files_joint = os.listdir(joint_matrices_path)
    dGGs = [f for f in files_joint if f.endswith('bindingmode_2.csv')] #changed to bm 1

    dGGj = pd.read_csv(os.path.join(joint_matrices_path, dGGs[0]), index_col=0)

    dGGm = pd.read_csv(os.path.join(sep_matrices_path, meths[0]), index_col=0)
    dGGum = pd.read_csv(os.path.join(sep_matrices_path, unmeths[0]), index_col=0)


    ##############
    #Getting all the paths  for the raw data, looks ugly and it is
    SMSAGxx = latest_curation[latest_curation.TF == TF].exp.values[0]
    BC = metadata[
        (metadata['TF'] == TF) & 
        (metadata['experiment'].str.replace('exp', '').astype(int) == int(SMSAGxx.replace('exp', '')))].Chip_pos.values[0]
    put_in = metadata[
        (metadata['TF'] == TF) & 
        (metadata['experiment'].str.replace('exp', '').astype(int) == int(SMSAGxx.replace('exp', '')))].input_library.values[0]

    SMSAGxx = metadata_dict[SMSAGxx]


    mBC, umBC, input_path = assign_mBC(put_in)

    input_path = os.path.join(input_path, f'BC{BC}_contamination_filtered.csv')
    ########
    #load the stuff
    BC_in = pd.read_csv(input_path)

    df_in = pd.read_csv(
        f'/home/gralak/updepla/users/gralak/NAS2/SmileSeq_paper/SmileSeq_experiments/{SMSAGxx}/00_read_in_data/output/BC{BC}_contamination_filtered.csv'
    )

    in_m = BC_in[BC_in['methl'] == mBC].reset_index(drop=True)
    in_nm = BC_in[BC_in['methl'] == umBC].reset_index(drop=True)

    df_m = df_in[df_in['methl'] == mBC].reset_index(drop=True)
    df_nm = df_in[df_in['methl'] == umBC].reset_index(drop=True)

    #save all matrices in .T format. save as well consensus

    if meth_consensus is None or unmeth_consensus is None:
        write_sectioned_csv(
            os.path.join(data_path, f'{TF}_report_for_paper_bindingmode2.csv'),            
            [
                (f'{TF} joint ddG', dGGj.T),
                (f'{TF} methylated ddG', dGGm.T),
                (f'{TF} unmethylated ddG', dGGum.T),
                ('divergence series', div_series.T),
                ('ppms not divergent', pd.DataFrame()),
                
            ]
        )
    else:
        write_sectioned_csv(
            os.path.join(data_path, f'{TF}_report_for_paper_bindingmode2.csv'),            
            [
                (f'{TF} joint ddG', dGGj.T),
                (f'{TF} methylated ddG', dGGm.T),
                (f'{TF} unmethylated ddG', dGGum.T),
                ('divergence series', div_series.T),
                ('methylated consensus', meth_consensus.T),
                ('unmethylated consensus', unmeth_consensus.T)
            ]
        )

    

In [101]:
metadata = pd.read_csv('/data/gralak/meSMiLEseq_github/meSMiLEseq/metadata.csv')
latest_curation = pd.read_csv('/data/gralak/meSMiLEseq_github/meSMiLEseq/TF_with_motifs_latest_curation.csv')
TFs = os.listdir('/home/gralak/updepla/users/gralak/SmileSeq_paper/meSMiLEseq_motifs_and_scatterplots_for_publication')
filtered_TFs = [tf for tf in TFs if not tf.endswith('.csv')]

In [109]:
#################################################HERE

In [499]:
filtered_TFs = ['ZNF716_FL', 'PRDM10_DBD', 'ZNF614_FL', 'ZNF680_FL']

In [507]:
for protein in filtered_TFs:
    TF = protein
    if TF == 'ZNF716_FL':
        print('include bs2!')
    filtered_TFs.remove(protein)
    print(f'{len(filtered_TFs)} to go, courage!!! <3')
    break

print(TF)

0 to go, courage!!! <3
ZNF680_FL


In [508]:
data_path = f'/home/gralak/updepla/users/gralak/SmileSeq_paper/meSMiLEseq_motifs_and_scatterplots_for_publication/{TF}/'

joint_matrices_path = os.path.join(data_path, 'joint/matrices/')
sep_matrices_path = os.path.join(data_path, 'separated/matrices/')    

try:
    meth_consensus = pd.read_csv(os.path.join(data_path, 'methylated_consensus_ppm_logo_bindingmode_2.csv'))
except FileNotFoundError:
    meth_consensus = None

try:
    unmeth_consensus = pd.read_csv(os.path.join(data_path, 'unmethylated_consensus_ppm_logo_bindingmode_2.csv'))
except FileNotFoundError:
    unmeth_consensus = None

div_series = pd.read_csv(os.path.join(data_path, 'divergence_series_bindingmode2.csv'))



files_sep = os.listdir(sep_matrices_path)
meths = [f for f in files_sep if f.endswith('_methylated_bindingmode_2.csv')]
unmeths = [f for f in files_sep if f.endswith('_unmethylated_bindingmode_2.csv')]
#meths = [f for f in files_sep if f.endswith('_methylated_bindingmode_1.csv')]
#unmeths = [f for f in files_sep if f.endswith('_unmethylated_bindingmode_1.csv')]

files_joint = os.listdir(joint_matrices_path)
dGGs = [f for f in files_joint if f.endswith('bindingmode_2.csv')] #changed to bm 1

dGGj = pd.read_csv(os.path.join(joint_matrices_path, dGGs[0]), index_col=0)

dGGm = pd.read_csv(os.path.join(sep_matrices_path, meths[0]), index_col=0)
dGGum = pd.read_csv(os.path.join(sep_matrices_path, unmeths[0]), index_col=0)


##############
#Getting all the paths  for the raw data, looks ugly and it is
SMSAGxx = latest_curation[latest_curation.TF == TF].exp.values[0]
BC = metadata[
    (metadata['TF'] == TF) & 
    (metadata['experiment'].str.replace('exp', '').astype(int) == int(SMSAGxx.replace('exp', '')))].Chip_pos.values[0]
put_in = metadata[
    (metadata['TF'] == TF) & 
    (metadata['experiment'].str.replace('exp', '').astype(int) == int(SMSAGxx.replace('exp', '')))].input_library.values[0]

SMSAGxx = metadata_dict[SMSAGxx]


mBC, umBC, input_path = assign_mBC(put_in)

input_path = os.path.join(input_path, f'BC{BC}_contamination_filtered.csv')
########
#load the stuff
BC_in = pd.read_csv(input_path)

df_in = pd.read_csv(
    f'/home/gralak/updepla/users/gralak/NAS2/SmileSeq_paper/SmileSeq_experiments/{SMSAGxx}/00_read_in_data/output/BC{BC}_contamination_filtered.csv'
)

in_m = BC_in[BC_in['methl'] == mBC].reset_index(drop=True)
in_nm = BC_in[BC_in['methl'] == umBC].reset_index(drop=True)

df_m = df_in[df_in['methl'] == mBC].reset_index(drop=True)
df_nm = df_in[df_in['methl'] == umBC].reset_index(drop=True)

if meth_consensus is None or unmeth_consensus is None:
    write_sectioned_csv(
        os.path.join(data_path, f'{TF}_permutations_bindingmode_2.csv'),            
                [
                    (f'{TF}, no differences between mLib and uLib ppms', pd.DataFrame()),
            ]
    )
    print('STOP RIGHT HERE, DONT CONTINUE! NEXT TF')
elif div_series.shape[1] != 3:
    write_sectioned_csv(
        os.path.join(data_path, f'{TF}_permutations_bindingmode_2.csv'),            
                [
                    (f'{TF}, no differences between mLib and uLib ppms', pd.DataFrame()),
            ]
    )
    print('STOP RIGHT HERE, DONT CONTINUE! NEXT TF')




STOP RIGHT HERE, DONT CONTINUE! NEXT TF


In [485]:
check_m = utils.ddg_to_consensus(meth_consensus)
check_um = utils.ddg_to_consensus(unmeth_consensus)

cons_m = check_m
cons_um = check_um

In [486]:
CGm = utils.where_CG(cons_m)
CGum = utils.where_CG(cons_um)

CG = set(CGm + CGum)

c_js = div_series.JensenShannon_divergence_mlib_ulib.dropna()

js_div = list(c_js[c_js >= 0.01].index)

differences = utils.resolve_cg_js_div(CG=CG, js_div=js_div)

print(CG)
print(cons_m)
print(cons_um)
print(differences)



{2, 4}
GGCGCGGCCATGG
GTCGCGGCCATGG
[2, 4, 5, 9]


In [ ]:
if len(differences) == 0:
    write_sectioned_csv(
        os.path.join(data_path, f'{TF}_permutations_bindingmode_2.csv'),            
                [
                    (f'{TF}, no differences between mLib and uLib ppms', pd.DataFrame()),
            ]
    )
    print('STOP RIGHT HERE, DONT CONTINUE! NEXT TF')    

In [475]:
#differences = [3]

In [487]:
def find_consensus_coords(a: pd.DataFrame, b: pd.DataFrame):
    # decide which is longer
    if len(a) >= len(b):
        long_df, short_df = a, b
    else:
        long_df, short_df = b, a

    cons_long = utils.ddg_to_consensus(long_df)
    cons_short = utils.ddg_to_consensus(short_df)

    start = cons_long.find(cons_short)
    if start == -1:
        return []   # no match

    # list of indices, works directly with .iloc
    return list(range(start, start + len(short_df)))

In [488]:
diff = utils.hamming_distance(cons_m, cons_um)
if diff > 1:
    consensus_seqs = [cons_m, cons_um]
else:
    consensus_seqs = [cons_m]


limits_m = find_consensus_coords(dGGm, meth_consensus)
limits_um = find_consensus_coords(dGGum, unmeth_consensus)

# What if there are secondary CGs that aer not picked up by consensus?
check_secondary_CG_sequences_m = utils.find_stabilizing_CG(dGGm.loc[limits_m,:].reset_index(drop=True))
check_secondary_CG_sequences_um = utils.find_stabilizing_CG(dGGum.loc[limits_um,:].reset_index(drop=True))

for id in check_secondary_CG_sequences_m:
    if not cons_m[id] == 'C' and not cons_m[id+1] == 'G':
        sec_cons_m = list(cons_m)
        sec_cons_m[id] = 'C'
        sec_cons_m[id+1] = 'G'
        sec_cons_m = ''.join(sec_cons_m)
        consensus_seqs.append(sec_cons_m)

for id in check_secondary_CG_sequences_um:
    if not cons_um[id] == 'C' and not cons_um[id+1] == 'G':
        sec_cons_um = list(cons_um)
        sec_cons_um[id] = 'C'
        sec_cons_um[id+1] = 'G'
        sec_cons_um = ''.join(sec_cons_um)
        consensus_seqs.append(sec_cons_um)

print(consensus_seqs)

['GGCGCGGCCATGG']


In [478]:
#consensus_seqs = ['GTAAGATTG', 'GTAAGACGG']
#differences = [7]

In [ ]:
printable_m = []
printable_um = []
rows_methylated = []
rows_unmethylated = []

for sequence in consensus_seqs:
    for posi in differences:
        if len(sequence) > 9:
            k = 7
            _k = k//2
            k_ = k - _k
            if posi - _k < 0:
                frag = sequence[0:k]
                p = posi
            elif posi + k_ >= len(sequence):
                frag = sequence[-k:]
                #last_index = len(sequence)
                p = posi - len(sequence) + k
            else:
                frag = sequence[posi-_k:posi+k_]
                p = _k
        else:
            k = len(sequence)
            frag = sequence
            p = posi

        search_for = []
        search_for.extend(utils.single_point_mut_seq(frag, p))
        search_for.append(frag)

        search_for_rev = [utils.rev_complement_seq(i) for i in search_for]
        
        search_for_rev.extend(search_for)

        pattern = '|'.join(search_for_rev)
        consens_m = df_m[df_m['random24'].str.contains(pattern, regex=True)]
        consens_um = df_nm[df_nm['random24'].str.contains(pattern, regex=True)]

        #consens_kmer_m = utils.kmer_counting(consens_m, kmer=k,one_df=True, mbc = [mBC, umBC])
        #consens_kmer_um = utils.kmer_counting(consens_um, kmer=k,one_df=True, mbc = [mBC, umBC])

        ###
        # input control

        m_in = in_m[in_m['random24'].str.contains(pattern, regex=True)]
        um_in = in_nm[in_nm['random24'].str.contains(pattern, regex=True)]

        #km_in = utils.kmer_counting(m_in, kmer=k,one_df=True, mbc = [mBC, umBC])
        #kum_in = utils.kmer_counting(um_in, kmer=k,one_df=True, mbc = [mBC, umBC])

        ###
        large_df_m = utils.kmer_counting(df=[m_in, consens_m],
                                        kmer=k,
                                        status=['input','eluted'],
                                        one_df=False,
                                        mbc=[mBC, umBC])
        
        large_df_um = utils.kmer_counting(df=[um_in, consens_um],
                                        kmer=k,
                                        status=['input','eluted'],
                                        one_df=False,
                                        mbc=[mBC, umBC])
                                

    ##################
        distrib_m = {}
        wrd_n = 0
        for word in search_for:
            select = large_df_m[large_df_m['kmer'].str.contains(word, regex=True)]
            select_rev = large_df_m[large_df_m['kmer'].str.contains(utils.rev_complement_seq(word), regex=True)]
            freq = select[select['status'] == 'eluted']['count'].sum()/select[select['status'] == 'input']['count'].sum()
            freq += select_rev[select_rev['status'] == 'eluted']['count'].sum()/select_rev[select_rev['status'] == 'input']['count'].sum()
            wrd_n += freq

            distrib_m[word] = freq

        distrib_m['sum'] = wrd_n

        
        printable_m.append(f'methylated {sequence}, nucleotide {posi+1}:')
        context = f'methylated {sequence}'

        for word in search_for:
            fraction = distrib_m[word] / distrib_m['sum']
            printable_m.append(f"{word}: {fraction:.3f}, abs occurrences {distrib_m[word]}")
            rows_methylated.append({
                'consensus': context,
                'position': posi+1,
                'search for': word,
                'value': round(fraction, 3),
                'norm. occurrences': distrib_m[word]
            })

        

        

    ###################
        distrib_um = {}
        wrd_n = 0
        for word in search_for:
            select = large_df_um[large_df_um['kmer'].str.contains(word, regex=True)]
            select_rev = large_df_um[large_df_um['kmer'].str.contains(utils.rev_complement_seq(word), regex=True)]
            freq = select[select['status'] == 'eluted']['count'].sum()/select[select['status'] == 'input']['count'].sum()
            freq += select_rev[select_rev['status'] == 'eluted']['count'].sum()/select_rev[select_rev['status'] == 'input']['count'].sum()
            wrd_n += freq
            distrib_um[word] = freq

        distrib_um['sum'] = wrd_n

        
        printable_um.append(f'unmethylated {sequence}, nucleotide {posi+1}:')
        context = f'unmethylated {sequence}'
        
        for word in search_for:
            fraction = distrib_um[word] / distrib_um['sum']
            printable_um.append(f"{word}: {fraction:.3f}, abs occurrences {distrib_um[word]}")
            rows_unmethylated.append({
                'consensus': context,
                'position': posi+1,
                'search for': word,
                'value': round(fraction, 3),
                'norm. occurrences': distrib_um[word]
            })



meth_df = pd.DataFrame(rows_methylated)
unmeth_df = pd.DataFrame(rows_unmethylated)

write_sectioned_csv(
    os.path.join(data_path, f'{TF}_permutations_bindingmode_2.csv'),            
            [
                (f'{TF}, mLib permutations', meth_df),
                (f'{TF}, uLib permutations', unmeth_df),
        ]
    )


#meth_df.to_csv(os.path.join(data_path, "methylated_permutations_bindingmode_1.csv"), index=False)
#unmeth_df.to_csv(os.path.join(data_path, "unmethylated_permutations_bindingmode_1.csv"), index=False)

In [371]:
TF

'TPRX1_DBD'

In [2]:
base = '/home/gralak/updepla/users/gralak/SmileSeq_paper/meSMiLEseq_motifs_and_scatterplots_for_publication/'
TF = 'USF3_DBD/'

In [3]:
meth_consensus = pd.read_csv(os.path.join(base, TF, 'methylated_consensus_ppm_logo.csv'))
unmeth_consensus = pd.read_csv(os.path.join(base, TF, 'unmethylated_consensus_ppm_logo.csv'))

In [ ]:

cons_um = 

,A,C,G,T
0,0.303757,0.204723,0.211843,0.279677
1,0.239967,0.279134,0.248631,0.232268
2,0.280562,0.313827,0.199493,0.206117
3,0.212568,0.251240,0.320103,0.216089
4,0.413362,0.162507,0.194600,0.229532
5,0.252185,0.394091,0.180774,0.172949
6,0.240805,0.214396,0.298256,0.246544
7,0.209485,0.205935,0.262937,0.321643
8,0.247083,0.213386,0.323054,0.216476
9,0.239680,0.228690,0.321115,0.210515
